# Notebook 8: Confidence Metrics and Interpretation of AlphaFold2 Outputs

**Series: The Mathematics and Architecture of AlphaFold2 (8 of 8)**

---

## Objective

Learn to read and interpret AlphaFold2 outputs -- pLDDT, PAE, pTM scores -- and understand what they mean for your predictions.

AlphaFold2 does not merely predict a 3D structure; it provides a rich set of confidence metrics that quantify *how much you should trust* different aspects of the prediction. Misinterpreting these metrics is one of the most common mistakes in structural biology today. This notebook provides the mathematical foundations and practical intuition needed to read AF2 outputs correctly.

---

**Prerequisites:** Notebooks 1--7 of this series (MSA processing, Evoformer, IPA, structure module, loss functions).

**Dependencies:** `numpy`, `matplotlib`, `mpl_toolkits.mplot3d`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, ArrowStyle
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Notebook 8: Confidence Metrics and Interpretation")
print("All data in this notebook is synthetic for pedagogical purposes.")

---

## 1. AlphaFold2 Outputs Overview

For each input sequence, AlphaFold2 produces a comprehensive set of outputs:

| Output | Shape | Description |
|--------|-------|-------------|
| **3D coordinates** | $(L, 37, 3)$ | All-atom coordinates for $L$ residues, 37 atom types |
| **pLDDT** | $(L,)$ | Per-residue confidence score in $[0, 100]$ |
| **PAE matrix** | $(L, L)$ | Predicted aligned error between all residue pairs |
| **pTM score** | scalar | Predicted TM-score summarizing global fold quality |
| **Distogram** | $(L, L, 64)$ | Predicted distance distribution between $C_\beta$ atoms |
| **5 ranked models** | list | Models from different random seeds, ranked by confidence |

Each of these outputs addresses a different question about prediction quality:

- **pLDDT** answers: "How accurate is the *local* structure around each residue?"
- **PAE** answers: "How well do we know the *relative position* of residue $i$ with respect to residue $j$?"
- **pTM** answers: "How correct is the *overall fold topology*?"

Understanding the distinction between these is critical for responsible use of AF2 predictions.

In [ ]:
# --- Synthetic protein output card ---
# Generate a synthetic 150-residue protein with all AF2 output metrics

L = 150
residues = np.arange(1, L + 1)

# Synthetic pLDDT: well-folded core with a loop and disordered tail
plddt = np.zeros(L)
plddt[:40] = np.random.uniform(88, 96, 40)        # Domain 1 (well-folded)
plddt[40:55] = np.random.uniform(55, 70, 15)      # Flexible linker
plddt[55:120] = np.random.uniform(85, 95, 65)     # Domain 2 (well-folded)
plddt[120:135] = np.random.uniform(45, 65, 15)    # Loop
plddt[135:] = np.random.uniform(20, 40, 15)       # Disordered tail
# Smooth transitions
kernel = np.ones(5) / 5
plddt_smooth = np.convolve(plddt, kernel, mode='same')
plddt_smooth = np.clip(plddt_smooth, 0, 100)

# Synthetic PAE matrix
pae = np.full((L, L), 20.0)
# Domain 1 block
pae[:40, :40] = np.random.uniform(1.5, 4.0, (40, 40))
# Domain 2 block
pae[55:120, 55:120] = np.random.uniform(1.5, 4.5, (65, 65))
# Linker region: moderate
pae[40:55, 40:55] = np.random.uniform(6, 12, (15, 15))
# Inter-domain: partially confident
pae[:40, 55:120] = np.random.uniform(8, 15, (40, 65))
pae[55:120, :40] = np.random.uniform(8, 15, (65, 40))
# Disordered tail: high error
pae[135:, :] = np.random.uniform(18, 30, (15, L))
pae[:, 135:] = np.random.uniform(18, 30, (L, 15))
np.fill_diagonal(pae, 0)

# Synthetic 3D backbone (helix + loop + helix + loop + coil)
t = np.linspace(0, 12 * np.pi, L)
x = np.cumsum(np.cos(t * 0.3) * 0.5 + np.random.normal(0, 0.1, L))
y = np.cumsum(np.sin(t * 0.3) * 0.5 + np.random.normal(0, 0.1, L))
z = np.cumsum(np.sin(t * 0.15) * 0.3 + np.random.normal(0, 0.05, L))

# Model ranking scores
model_scores = [0.87, 0.91, 0.82, 0.89, 0.79]
model_names = ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5']
ranked_order = np.argsort(model_scores)[::-1]

# --- Create the output card ---
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# (1) 3D backbone colored by pLDDT
ax1 = fig.add_subplot(gs[0, 0], projection='3d')
norm = Normalize(vmin=0, vmax=100)
colors = cm.RdYlBu(norm(plddt_smooth))
for i in range(L - 1):
    ax1.plot(x[i:i+2], y[i:i+2], z[i:i+2], color=colors[i], linewidth=2)
ax1.set_title('3D Backbone (colored by pLDDT)', fontsize=13)
ax1.set_xlabel('x', fontsize=11)
ax1.set_ylabel('y', fontsize=11)
ax1.set_zlabel('z', fontsize=11)
ax1.tick_params(labelsize=9)

# (2) PAE heatmap
ax2 = fig.add_subplot(gs[0, 1])
im = ax2.imshow(pae, cmap='Greens', vmin=0, vmax=30, origin='lower', aspect='equal')
ax2.set_title('Predicted Aligned Error (PAE)', fontsize=13)
ax2.set_xlabel('Scored residue $j$', fontsize=11)
ax2.set_ylabel('Aligned residue $i$', fontsize=11)
plt.colorbar(im, ax=ax2, label='Expected error (\u00c5)', shrink=0.8)
ax2.tick_params(labelsize=9)

# (3) pLDDT line plot
ax3 = fig.add_subplot(gs[1, 0])
ax3.fill_between(residues, 0, 50, color='#FF7F7F', alpha=0.15, label='Very low (<50)')
ax3.fill_between(residues, 50, 70, color='#FFD700', alpha=0.15, label='Low (50-70)')
ax3.fill_between(residues, 70, 90, color='#87CEEB', alpha=0.15, label='Confident (70-90)')
ax3.fill_between(residues, 90, 100, color='#0066CC', alpha=0.15, label='Very high (>90)')
ax3.plot(residues, plddt_smooth, color='black', linewidth=1.5)
ax3.set_xlim(1, L)
ax3.set_ylim(0, 100)
ax3.set_xlabel('Residue index', fontsize=11)
ax3.set_ylabel('pLDDT', fontsize=11)
ax3.set_title('Per-Residue Confidence (pLDDT)', fontsize=13)
ax3.legend(fontsize=9, loc='lower left')
ax3.tick_params(labelsize=9)

# (4) Model ranking bar chart
ax4 = fig.add_subplot(gs[1, 1])
bar_colors = ['#4CAF50' if i == ranked_order[0] else '#90CAF9' for i in range(5)]
bars = ax4.bar(model_names, model_scores, color=bar_colors, edgecolor='gray', linewidth=0.8)
ax4.set_ylabel('Model confidence score', fontsize=11)
ax4.set_title('Model Ranking (5 seeds)', fontsize=13)
ax4.set_ylim(0.5, 1.0)
ax4.axhline(y=model_scores[ranked_order[0]], color='green', linestyle='--', alpha=0.5)
ax4.annotate('Top-ranked', xy=(ranked_order[0], model_scores[ranked_order[0]]),
             xytext=(ranked_order[0] + 0.3, model_scores[ranked_order[0]] + 0.03),
             fontsize=11, color='green',
             arrowprops=dict(arrowstyle='->', color='green'))
ax4.tick_params(labelsize=9)

fig.suptitle('AlphaFold2 Output Card: Synthetic 150-Residue Protein', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---

## 2. pLDDT: Per-Residue Confidence

### 2.1 Definition and Mathematical Formulation

The **predicted Local Distance Difference Test (pLDDT)** is AF2's estimate of the LDDT score for each residue. The LDDT metric measures how well local interatomic distances are preserved between the predicted and true structures.

For each residue $i$, the true LDDT considers all atoms within 15 \AA\ of residue $i$'s $C_\alpha$ in the true structure and computes the fraction of pairwise distances that are predicted within tolerance thresholds (0.5, 1, 2, 4 \AA).

AF2 predicts pLDDT as a **50-bin classification problem**. The network outputs a probability distribution $\{p_b\}_{b=1}^{50}$ over 50 bins, where each bin corresponds to an LDDT range of width $1/50 = 0.02$. The predicted pLDDT is the expected value:

$$
\text{pLDDT}_i = 100 \times \sum_{b=1}^{50} \frac{b}{50} \, p_b^{(i)}
$$

where $p_b^{(i)}$ is the predicted probability that residue $i$'s true LDDT falls in bin $b$, and the factor of 100 scales the result to the conventional 0--100 range.

### 2.2 Confidence Categories

The AF2 team established four confidence categories:

| pLDDT Range | Category | Interpretation |
|------------|----------|----------------|
| $> 90$ | Very high | Backbone and side-chain positions highly reliable |
| $70$--$90$ | Confident | Backbone reliable, some side-chain uncertainty |
| $50$--$70$ | Low | Backbone approximate, side-chains unreliable |
| $< 50$ | Very low | Should not be interpreted structurally; often disordered |

A critical insight: **disordered regions typically have pLDDT $< 50$**, not because AF2 fails to predict them, but because there is no single well-defined structure to predict. AF2 effectively becomes a disorder predictor in these regions.

In [ ]:
# --- Synthetic 200-residue protein with varying confidence regions ---

L2 = 200
residues2 = np.arange(1, L2 + 1)

# Build pLDDT profile with distinct regions
plddt2 = np.zeros(L2)

# Well-folded core: residues 1-80 (pLDDT 85-95)
plddt2[:80] = np.random.uniform(85, 95, 80)
# Add some structure: helices have higher confidence
for start in [5, 25, 50, 65]:
    end = min(start + 12, 80)
    plddt2[start:end] += np.random.uniform(2, 5, end - start)

# Flexible loop: residues 81-120 (pLDDT 40-60)
plddt2[80:120] = np.random.uniform(40, 60, 40)
# Gradual transition at boundaries
for i in range(5):
    plddt2[80 + i] = plddt2[79] - (plddt2[79] - 55) * (i + 1) / 6
    plddt2[119 - i] = plddt2[120] if 120 < L2 else 50

# Second folded region: residues 121-160 (pLDDT 85-95)
plddt2[120:160] = np.random.uniform(85, 95, 40)

# Disordered tail: residues 161-200 (pLDDT 20-40)
plddt2[160:] = np.random.uniform(20, 40, 40)
# Transition
for i in range(8):
    plddt2[160 + i] = plddt2[159] - (plddt2[159] - 30) * (i + 1) / 9

plddt2 = np.clip(plddt2, 0, 100)

# Smooth slightly
kernel = np.ones(3) / 3
plddt2 = np.convolve(plddt2, kernel, mode='same')
plddt2 = np.clip(plddt2, 0, 100)

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 6))

# Confidence zone bands
ax.axhspan(0, 50, color='#FF6B6B', alpha=0.12, label='Very low (< 50)')
ax.axhspan(50, 70, color='#FFA500', alpha=0.12, label='Low (50--70)')
ax.axhspan(70, 90, color='#87CEEB', alpha=0.12, label='Confident (70--90)')
ax.axhspan(90, 100, color='#1E90FF', alpha=0.12, label='Very high (> 90)')

# Threshold lines
for threshold in [50, 70, 90]:
    ax.axhline(y=threshold, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)

# Color the line by pLDDT value
cmap_plddt = cm.RdYlBu
norm_plddt = Normalize(vmin=0, vmax=100)
for i in range(L2 - 1):
    color = cmap_plddt(norm_plddt(plddt2[i]))
    ax.plot(residues2[i:i+2], plddt2[i:i+2], color=color, linewidth=2.2)

# Region annotations
ax.annotate('Well-folded core', xy=(40, 92), fontsize=12, ha='center',
            color='#0055AA')
ax.annotate('Flexible loop', xy=(100, 62), fontsize=12, ha='center',
            color='#CC7700')
ax.annotate('Folded domain 2', xy=(140, 92), fontsize=12, ha='center',
            color='#0055AA')
ax.annotate('Disordered tail', xy=(180, 42), fontsize=12, ha='center',
            color='#CC0000')

ax.set_xlim(1, L2)
ax.set_ylim(0, 100)
ax.set_xlabel('Residue index', fontsize=12)
ax.set_ylabel('pLDDT', fontsize=12)
ax.set_title('pLDDT Profile: Synthetic 200-Residue Protein with Varying Confidence Regions', fontsize=14)
ax.legend(fontsize=11, loc='lower left', framealpha=0.9)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- 3D backbone colored by pLDDT with colorbar ---

# Generate a synthetic 3D backbone for the 200-residue protein
# Use a combination of helical and extended segments
np.random.seed(7)

coords = np.zeros((L2, 3))
# Well-folded core: compact helical bundle
for i in range(80):
    t = i * 0.3
    coords[i] = [5 * np.cos(t) + 0.5 * np.cos(0.05 * i),
                 5 * np.sin(t) + 0.5 * np.sin(0.05 * i),
                 1.5 * i / 80]

# Flexible loop: more extended
for i in range(80, 120):
    j = i - 80
    coords[i] = coords[79] + [(j + 1) * 0.4 + np.random.normal(0, 0.3),
                               (j + 1) * 0.3 + np.random.normal(0, 0.3),
                               (j + 1) * 0.2 + np.random.normal(0, 0.2)]

# Second folded domain: another compact region
for i in range(120, 160):
    t = (i - 120) * 0.35
    coords[i] = coords[119] + [3 * np.cos(t) + 5,
                                3 * np.sin(t),
                                1.2 * (i - 120) / 40]

# Disordered tail: random walk
for i in range(160, L2):
    step = np.random.normal(0, 0.8, 3)
    step[0] += 0.5  # general direction
    coords[i] = coords[i - 1] + step

x3, y3, z3 = coords[:, 0], coords[:, 1], coords[:, 2]

# --- 3D Plot ---
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

cmap_3d = cm.RdYlBu
norm_3d = Normalize(vmin=0, vmax=100)

# Plot backbone segments
for i in range(L2 - 1):
    color = cmap_3d(norm_3d(plddt2[i]))
    ax.plot(x3[i:i+2], y3[i:i+2], z3[i:i+2], color=color, linewidth=2.5)

# Scatter points at key residues for emphasis
sc = ax.scatter(x3[::10], y3[::10], z3[::10], c=plddt2[::10], cmap='RdYlBu',
                vmin=0, vmax=100, s=40, edgecolors='gray', linewidth=0.5, zorder=5)

# Colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)
cbar.set_label('pLDDT', fontsize=12)
cbar.ax.tick_params(labelsize=10)

ax.set_title('3D Backbone Colored by pLDDT\n(Blue = High confidence, Red = Low confidence)', fontsize=13)
ax.set_xlabel('x (\u00c5)', fontsize=11)
ax.set_ylabel('y (\u00c5)', fontsize=11)
ax.set_zlabel('z (\u00c5)', fontsize=11)
ax.tick_params(labelsize=9)
ax.view_init(elev=20, azim=135)

plt.tight_layout()
plt.show()

---

## 3. PAE: Predicted Aligned Error

### 3.1 Definition

The **Predicted Aligned Error (PAE)** provides pairwise confidence information. For each pair of residues $(i, j)$, the PAE answers the question:

> *If we align the predicted structure to the true structure using residue $j$'s local reference frame, what is the expected positional error at residue $i$?*

Formally, let $T_j^{\text{true}}$ and $T_j^{\text{pred}}$ be the rigid-body frames at residue $j$ in the true and predicted structures, respectively. Let $\hat{\mathbf{x}}_i$ be the predicted position of residue $i$ and $\mathbf{x}_i^{\text{true}}$ its true position. Then:

$$
\text{PAE}(i, j) = \mathbb{E}\left[\left\| \left(T_j^{\text{true}}\right)^{-1} \circ \hat{\mathbf{x}}_i \;-\; \left(T_j^{\text{true}}\right)^{-1} \circ \mathbf{x}_i^{\text{true}} \right\|\right]
$$

### 3.2 Prediction Mechanism

PAE is predicted as a **64-bin classification** over the range $[0, 32]$ \AA. The network outputs a probability distribution $\{q_b^{(i,j)}\}_{b=1}^{64}$ for each residue pair, and the expected error is:

$$
\text{PAE}(i, j) = \sum_{b=1}^{64} \frac{b \times 32}{64} \, q_b^{(i,j)} = \sum_{b=1}^{64} \frac{b}{2} \, q_b^{(i,j)}
$$

### 3.3 Interpreting PAE Matrices

Key patterns in PAE matrices:

- **Low-PAE diagonal blocks**: Indicate rigid domains where internal structure is well-predicted
- **Low off-diagonal blocks**: Indicate that the relative orientation between two domains/chains is well-predicted
- **High off-diagonal regions**: The relative positioning of those residue pairs is uncertain
- **PAE is NOT symmetric** in general: $\text{PAE}(i,j) \neq \text{PAE}(j,i)$ because alignment at $j$ vs alignment at $i$ probes different aspects of the structure

In [ ]:
# --- Synthetic PAE matrices: confident vs uncertain inter-domain orientation ---

L3 = 150
domain1_end = 80
domain2_start = 81

def make_pae_matrix(L, d1_end, d2_start, inter_domain_low=False):
    """Generate synthetic PAE matrix for a 2-domain protein."""
    pae = np.full((L, L), 25.0)  # default high PAE
    
    # Domain 1 internal: low PAE
    for i in range(d1_end):
        for j in range(d1_end):
            dist = abs(i - j)
            pae[i, j] = np.random.uniform(1.0, 3.0) + 0.02 * dist
    
    # Domain 2 internal: low PAE
    for i in range(d2_start - 1, L):
        for j in range(d2_start - 1, L):
            dist = abs(i - j)
            pae[i, j] = np.random.uniform(1.0, 3.5) + 0.02 * dist
    
    if inter_domain_low:
        # Well-predicted inter-domain relationship
        pae[:d1_end, d2_start-1:] = np.random.uniform(3.0, 7.0, (d1_end, L - d2_start + 1))
        pae[d2_start-1:, :d1_end] = np.random.uniform(3.0, 7.0, (L - d2_start + 1, d1_end))
    else:
        # Uncertain inter-domain orientation
        pae[:d1_end, d2_start-1:] = np.random.uniform(15.0, 28.0, (d1_end, L - d2_start + 1))
        pae[d2_start-1:, :d1_end] = np.random.uniform(15.0, 28.0, (L - d2_start + 1, d1_end))
    
    np.fill_diagonal(pae, 0)
    return np.clip(pae, 0, 31)

pae_uncertain = make_pae_matrix(L3, domain1_end, domain2_start, inter_domain_low=False)
pae_confident = make_pae_matrix(L3, domain1_end, domain2_start, inter_domain_low=True)

# --- Side-by-side heatmaps ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Uncertain inter-domain
im1 = ax1.imshow(pae_uncertain, cmap='Greens', vmin=0, vmax=31, origin='lower', aspect='equal')
ax1.set_title('Uncertain Inter-Domain Orientation', fontsize=13)
ax1.set_xlabel('Scored residue $j$', fontsize=12)
ax1.set_ylabel('Aligned residue $i$', fontsize=12)
ax1.axvline(x=domain1_end, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax1.axhline(y=domain1_end, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax1.annotate('Domain 1', xy=(25, 25), fontsize=11, color='white', ha='center')
ax1.annotate('Domain 2', xy=(115, 115), fontsize=11, color='white', ha='center')
ax1.annotate('High PAE:\nuncertain\nrelative pose', xy=(115, 30), fontsize=10,
             color='yellow', ha='center')
plt.colorbar(im1, ax=ax1, label='PAE (\u00c5)', shrink=0.8)
ax1.tick_params(labelsize=10)

# Confident inter-domain
im2 = ax2.imshow(pae_confident, cmap='Greens', vmin=0, vmax=31, origin='lower', aspect='equal')
ax2.set_title('Confident Inter-Domain Orientation', fontsize=13)
ax2.set_xlabel('Scored residue $j$', fontsize=12)
ax2.set_ylabel('Aligned residue $i$', fontsize=12)
ax2.axvline(x=domain1_end, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax2.axhline(y=domain1_end, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax2.annotate('Domain 1', xy=(25, 25), fontsize=11, color='white', ha='center')
ax2.annotate('Domain 2', xy=(115, 115), fontsize=11, color='white', ha='center')
ax2.annotate('Low PAE:\nconfident\nrelative pose', xy=(115, 30), fontsize=10,
             color='white', ha='center')
plt.colorbar(im2, ax=ax2, label='PAE (\u00c5)', shrink=0.8)
ax2.tick_params(labelsize=10)

fig.suptitle('PAE Matrices for a 2-Domain Protein (Residues 1--80 and 81--150)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Reading PAE Maps: A Practical Guide

PAE matrices encode rich structural information. With practice, you can read them like a map:

### Key Patterns

1. **Low diagonal blocks** = confident domains. Each contiguous block of low PAE along the diagonal corresponds to a structural domain whose internal conformation is well-predicted.

2. **Off-diagonal patterns** reveal inter-domain or inter-chain relationships. If the block at position $(\text{Domain}_A, \text{Domain}_B)$ has low PAE, the relative orientation of those domains is confident.

3. **Multimer PAE**: For multimeric predictions, the PAE matrix is organized by chain. Blocks along the diagonal correspond to intra-chain confidence; off-diagonal blocks correspond to inter-chain confidence.

4. **Asymmetry**: $\text{PAE}(i, j)$ measures error at $i$ when aligning at $j$'s frame. This is physically different from $\text{PAE}(j, i)$, although for well-predicted structures the matrix is approximately symmetric.

In [ ]:
# --- 4 synthetic PAE scenarios in a 2x2 grid ---

def make_single_domain_pae(L):
    """Single well-folded domain."""
    pae = np.zeros((L, L))
    for i in range(L):
        for j in range(L):
            dist = abs(i - j)
            pae[i, j] = np.random.uniform(0.5, 2.0) + 0.015 * dist
    np.fill_diagonal(pae, 0)
    return np.clip(pae, 0, 31)

def make_two_domain_uncertain(L, boundary):
    """Two confident domains, uncertain interface."""
    pae = np.full((L, L), 22.0)
    # Domain 1
    pae[:boundary, :boundary] = np.random.uniform(1.0, 3.5, (boundary, boundary))
    # Domain 2
    pae[boundary:, boundary:] = np.random.uniform(1.0, 3.5, (L - boundary, L - boundary))
    # Inter-domain: high PAE
    pae[:boundary, boundary:] = np.random.uniform(16, 26, (boundary, L - boundary))
    pae[boundary:, :boundary] = np.random.uniform(16, 26, (L - boundary, boundary))
    np.fill_diagonal(pae, 0)
    return np.clip(pae, 0, 31)

def make_two_domain_confident(L, boundary):
    """Two confident domains with confident interface."""
    pae = np.full((L, L), 5.0)
    pae[:boundary, :boundary] = np.random.uniform(0.8, 2.5, (boundary, boundary))
    pae[boundary:, boundary:] = np.random.uniform(0.8, 2.5, (L - boundary, L - boundary))
    pae[:boundary, boundary:] = np.random.uniform(3.0, 6.0, (boundary, L - boundary))
    pae[boundary:, :boundary] = np.random.uniform(3.0, 6.0, (L - boundary, boundary))
    np.fill_diagonal(pae, 0)
    return np.clip(pae, 0, 31)

def make_multimer_pae(L, chain_boundary):
    """Multimer with 2 chains, confident interaction."""
    pae = np.full((L, L), 8.0)
    # Chain A intra
    pae[:chain_boundary, :chain_boundary] = np.random.uniform(0.8, 3.0,
                                                               (chain_boundary, chain_boundary))
    # Chain B intra
    pae[chain_boundary:, chain_boundary:] = np.random.uniform(0.8, 3.0,
                                                               (L - chain_boundary, L - chain_boundary))
    # Inter-chain: confident interaction (low PAE for interface residues)
    # Interface residues: near the boundary
    interface_A = slice(chain_boundary - 25, chain_boundary)
    interface_B = slice(chain_boundary, chain_boundary + 25)
    pae[interface_A, interface_B] = np.random.uniform(3.0, 6.0, (25, 25))
    pae[interface_B, interface_A] = np.random.uniform(3.0, 6.0, (25, 25))
    # Non-interface inter-chain: moderate
    pae[:chain_boundary - 25, chain_boundary:] = np.random.uniform(8, 14,
                                                                    (chain_boundary - 25, L - chain_boundary))
    pae[chain_boundary + 25:, :chain_boundary] = np.random.uniform(8, 14,
                                                                    (L - chain_boundary - 25, chain_boundary))
    np.fill_diagonal(pae, 0)
    return np.clip(pae, 0, 31)

L_demo = 150
boundary = 80

pae_scenarios = [
    (make_single_domain_pae(L_demo), 'Single well-folded domain\n(uniformly low PAE)'),
    (make_two_domain_uncertain(L_demo, boundary), 'Two domains, uncertain interface\n(high off-diagonal PAE)'),
    (make_two_domain_confident(L_demo, boundary), 'Two domains, confident interface\n(low off-diagonal PAE)'),
    (make_multimer_pae(L_demo, boundary), 'Multimer (chains A + B)\nwith confident interaction')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 13))

for idx, (ax, (pae_mat, title)) in enumerate(zip(axes.flat, pae_scenarios)):
    im = ax.imshow(pae_mat, cmap='Greens', vmin=0, vmax=31, origin='lower', aspect='equal')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Scored residue $j$', fontsize=11)
    ax.set_ylabel('Aligned residue $i$', fontsize=11)
    ax.tick_params(labelsize=9)
    plt.colorbar(im, ax=ax, label='PAE (\u00c5)', shrink=0.8)
    
    # Add boundary lines for multi-domain/chain scenarios
    if idx >= 1:
        ax.axvline(x=boundary, color='white', linestyle='--', linewidth=1.2, alpha=0.7)
        ax.axhline(y=boundary, color='white', linestyle='--', linewidth=1.2, alpha=0.7)
    if idx == 3:
        ax.annotate('Chain A', xy=(30, 10), fontsize=10, color='white')
        ax.annotate('Chain B', xy=(100, 140), fontsize=10, color='white')
        ax.annotate('Interface', xy=(95, 62), fontsize=9, color='yellow')

fig.suptitle('Reading PAE Maps: Four Common Scenarios', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

## 5. pTM Score: Global Model Quality

### 5.1 TM-Score Background

The **Template Modeling score (TM-score)** is a length-independent metric for comparing protein structures. Given a predicted structure with residue positions $\hat{\mathbf{x}}_i$ and true structure with positions $\mathbf{x}_i^{\text{true}}$, after optimal superposition:

$$
\text{TM} = \frac{1}{L} \sum_{i=1}^{L} \frac{1}{1 + \left(\frac{d_i}{d_0(L)}\right)^2}
$$

where:
- $d_i = \|\hat{\mathbf{x}}_i - \mathbf{x}_i^{\text{true}}\|$ is the distance between corresponding $C_\alpha$ atoms after superposition
- $d_0(L) = 1.24 \sqrt[3]{L - 15} - 1.8$ is a length-dependent normalization that makes TM-score length-independent

### 5.2 Interpretation

| TM-score | Interpretation |
|----------|---------------|
| $> 0.5$ | Correct overall fold topology (same SCOP fold) |
| $0.17$ | Expected for random structure pairs |
| $> 0.7$ | High-confidence structural match |
| $\approx 1.0$ | Near-identical structures |

The critical threshold is **TM $> 0.5$**, which has been shown to reliably distinguish correct fold topology from incorrect.

### 5.3 pTM and ipTM

AF2 predicts TM-score (**pTM**) from the PAE logits using a differentiable approximation. For multimers, it also predicts **ipTM** (interface pTM), which considers only inter-chain residue pairs, focusing on whether the interface geometry is correct.

In [ ]:
# --- TM-score as a function of RMSD for different protein lengths ---

def tm_score_from_rmsd(rmsd, L, fraction_well_aligned=1.0):
    """
    Approximate TM-score assuming all residues have the same deviation (RMSD).
    In reality, TM-score is a sum over individual residue distances, but this
    gives the right qualitative behavior.
    """
    d0 = 1.24 * (L - 15) ** (1.0 / 3.0) - 1.8
    d0 = max(d0, 0.5)  # safety floor
    # Assume a distribution of per-residue distances with mean ~ rmsd
    # For simplicity: all residues at distance = rmsd
    tm = 1.0 / (1.0 + (rmsd / d0) ** 2)
    return tm

lengths = [100, 200, 400]
colors_tm = ['#1f77b4', '#ff7f0e', '#2ca02c']
rmsd_range = np.linspace(0, 20, 500)

fig, ax = plt.subplots(figsize=(11, 7))

for L_val, col in zip(lengths, colors_tm):
    tm_values = [tm_score_from_rmsd(r, L_val) for r in rmsd_range]
    d0 = 1.24 * (L_val - 15) ** (1.0 / 3.0) - 1.8
    ax.plot(rmsd_range, tm_values, color=col, linewidth=2.2,
            label=f'$L = {L_val}$ ($d_0 = {d0:.2f}$ \u00c5)')

# Threshold line
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.annotate('TM = 0.5 threshold\n(correct fold topology)',
            xy=(12, 0.52), fontsize=11, color='red')

# Annotate regions
ax.axhspan(0.7, 1.0, color='#1E90FF', alpha=0.06)
ax.axhspan(0.5, 0.7, color='#90EE90', alpha=0.06)
ax.axhspan(0.0, 0.5, color='#FFB6C1', alpha=0.06)

ax.annotate('High confidence (TM > 0.7)', xy=(0.5, 0.85), fontsize=11, color='#0055AA')
ax.annotate('Correct fold (0.5 < TM < 0.7)', xy=(0.5, 0.58), fontsize=11, color='#228B22')
ax.annotate('Incorrect fold (TM < 0.5)', xy=(0.5, 0.25), fontsize=11, color='#CC3333')

ax.set_xlabel('RMSD (\u00c5)', fontsize=12)
ax.set_ylabel('TM-score', fontsize=12)
ax.set_title('TM-Score vs. RMSD for Different Protein Lengths\n'
             r'$\mathrm{TM} = \frac{1}{1 + (d/d_0)^2}$, '
             r'$d_0 = 1.24\sqrt[3]{L-15} - 1.8$', fontsize=13)
ax.set_xlim(0, 20)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, loc='upper right')
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()

---

## 6. Model Ranking and Selection

### 6.1 Multiple Models from Multiple Seeds

AlphaFold2 generates **5 models** using different random seeds for the recycling and dropout stages. This ensemble serves two purposes:

1. **Model selection**: The best model is chosen based on a confidence metric.
2. **Uncertainty estimation**: Diversity among models indicates prediction uncertainty.

### 6.2 Ranking Criteria

For **monomers**, models are ranked by their mean pLDDT:

$$
\text{score}_{\text{monomer}} = \frac{1}{L} \sum_{i=1}^{L} \text{pLDDT}_i
$$

For **multimers**, a composite score is used:

$$
\text{score}_{\text{multimer}} = 0.8 \times \text{ipTM} + 0.2 \times \text{pTM}
$$

The weighting emphasizes interface quality (ipTM), since correctly predicting inter-chain contacts is typically the most challenging and most important aspect of multimer prediction.

In [ ]:
# --- Model ranking bar chart with confidence intervals ---

np.random.seed(99)

# Simulate 5 model scores and per-residue pLDDT distributions
n_models = 5
mean_plddts = [82.3, 87.1, 79.5, 85.8, 76.2]
std_plddts = [8.5, 6.2, 10.1, 7.0, 12.3]

model_labels = [f'Model {i+1}' for i in range(n_models)]
ranked_idx = np.argsort(mean_plddts)[::-1]
best_model = ranked_idx[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Left panel: bar chart with error bars ---
bar_colors = ['#4CAF50' if i == best_model else '#90CAF9' for i in range(n_models)]
bars = ax1.bar(model_labels, mean_plddts, yerr=std_plddts, capsize=5,
               color=bar_colors, edgecolor='gray', linewidth=0.8, error_kw={'linewidth': 1.5})

ax1.set_ylabel('Mean pLDDT', fontsize=12)
ax1.set_title('Model Ranking by Mean pLDDT (Monomer)', fontsize=13)
ax1.set_ylim(50, 100)
ax1.axhline(y=mean_plddts[best_model], color='green', linestyle='--', alpha=0.4)

# Annotate best model
ax1.annotate(f'Rank 1 (selected)\npLDDT = {mean_plddts[best_model]:.1f}',
             xy=(best_model, mean_plddts[best_model] + std_plddts[best_model] + 1),
             fontsize=11, ha='center', color='green')

# Show rank order
for rank, idx in enumerate(ranked_idx):
    ax1.text(idx, 53, f'Rank {rank + 1}', ha='center', fontsize=10, color='#555555')

ax1.tick_params(labelsize=11)

# --- Right panel: model diversity as uncertainty indicator ---
# Simulate per-residue pLDDT for each model
L_div = 100
residues_div = np.arange(1, L_div + 1)
model_plddts = []
base_profile = np.zeros(L_div)
base_profile[:40] = 90
base_profile[40:60] = 55  # uncertain region
base_profile[60:] = 88

for i in range(n_models):
    noise = np.random.normal(0, 2, L_div)
    # Add more variation in the uncertain region
    noise[40:60] += np.random.normal(0, 8, 20)
    profile = base_profile + noise + np.random.normal(mean_plddts[i] - 83, 1)
    model_plddts.append(np.clip(profile, 0, 100))

model_plddts = np.array(model_plddts)
mean_profile = model_plddts.mean(axis=0)
std_profile = model_plddts.std(axis=0)

for i in range(n_models):
    ax2.plot(residues_div, model_plddts[i], alpha=0.4, linewidth=1,
             label=f'Model {i+1}' if i < 3 else None)

ax2.plot(residues_div, mean_profile, color='black', linewidth=2, label='Mean')
ax2.fill_between(residues_div, mean_profile - std_profile, mean_profile + std_profile,
                 alpha=0.2, color='gray', label='$\\pm 1\\sigma$ (model diversity)')

ax2.axhspan(40, 60, xmin=0.38, xmax=0.62, color='red', alpha=0.08)
ax2.annotate('High model diversity\n= uncertain region', xy=(50, 42),
             fontsize=11, ha='center', color='#CC0000')

ax2.set_xlabel('Residue index', fontsize=12)
ax2.set_ylabel('pLDDT', fontsize=12)
ax2.set_title('Model Diversity Indicates Uncertainty', fontsize=13)
ax2.set_xlim(1, L_div)
ax2.set_ylim(0, 100)
ax2.legend(fontsize=10, loc='lower right')
ax2.tick_params(labelsize=11)

plt.tight_layout()
plt.show()

---

## 7. Disorder Prediction from pLDDT

### 7.1 AF2 as an "Accidental" Disorder Predictor

One of the most remarkable findings about AlphaFold2 is that it serves as a highly accurate predictor of **intrinsically disordered regions (IDRs)**. Regions with pLDDT $< 50$ correlate strongly with experimentally characterized disorder.

This is not a bug but a natural consequence of the prediction task. For disordered residues, there is no single well-defined 3D structure. The network correctly expresses low confidence because the concept of a "correct" position is ill-defined for these residues.

Importantly, AF2 often predicts disordered regions as extended, unstructured coils -- this is an artifact of the prediction, not a statement about the ensemble of conformations these regions actually adopt.

### 7.2 Practical Use

A simple threshold of pLDDT $< 50$ provides disorder annotations that are competitive with dedicated disorder predictors like IUPred and MobiDB-lite.

In [ ]:
# --- Disorder prediction from pLDDT ---

np.random.seed(21)
L_dis = 250
residues_dis = np.arange(1, L_dis + 1)

# Ground-truth disorder annotation (binary)
# Folded: residues 1-60, 100-180
# Disordered: residues 61-99 (IDR linker), 181-250 (IDR tail)
disorder_gt = np.zeros(L_dis)
disorder_gt[60:100] = 1.0    # IDR linker
disorder_gt[180:] = 1.0      # IDR C-terminal tail

# Synthetic pLDDT that correlates with disorder
plddt_dis = np.zeros(L_dis)
# Folded regions: high pLDDT
plddt_dis[:60] = np.random.uniform(82, 96, 60)
plddt_dis[100:180] = np.random.uniform(80, 94, 80)
# Disordered regions: low pLDDT
plddt_dis[60:100] = np.random.uniform(22, 48, 40)
plddt_dis[180:] = np.random.uniform(18, 42, 70)

# Add some noise and smooth transitions
for boundary in [60, 100, 180]:
    for k in range(4):
        if boundary + k < L_dis and boundary - k - 1 >= 0:
            plddt_dis[boundary + k] = (plddt_dis[boundary - k - 1] + plddt_dis[boundary + k]) / 2

kernel_dis = np.ones(3) / 3
plddt_dis = np.convolve(plddt_dis, kernel_dis, mode='same')
plddt_dis = np.clip(plddt_dis, 0, 100)

# Predicted disorder from pLDDT < 50
disorder_pred = (plddt_dis < 50).astype(float)

# --- Plot ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                                      gridspec_kw={'height_ratios': [3, 1, 1], 'hspace': 0.08})

# Top: pLDDT profile
ax1.axhline(y=50, color='red', linestyle='--', linewidth=1.2, alpha=0.6, label='Disorder threshold (pLDDT = 50)')
cmap_dis = cm.RdYlBu
norm_dis = Normalize(vmin=0, vmax=100)
for i in range(L_dis - 1):
    color = cmap_dis(norm_dis(plddt_dis[i]))
    ax1.plot(residues_dis[i:i+2], plddt_dis[i:i+2], color=color, linewidth=2)

ax1.set_ylabel('pLDDT', fontsize=12)
ax1.set_ylim(0, 100)
ax1.set_title('Disorder Prediction from pLDDT: Synthetic 250-Residue Protein', fontsize=14)
ax1.legend(fontsize=11, loc='lower right')
ax1.tick_params(labelsize=11)
ax1.annotate('Folded domain 1', xy=(30, 92), fontsize=11, ha='center', color='#0055AA')
ax1.annotate('IDR linker', xy=(80, 15), fontsize=11, ha='center', color='#CC0000')
ax1.annotate('Folded domain 2', xy=(140, 92), fontsize=11, ha='center', color='#0055AA')
ax1.annotate('IDR tail', xy=(215, 15), fontsize=11, ha='center', color='#CC0000')

# Middle: ground-truth disorder
ax2.fill_between(residues_dis, 0, disorder_gt, color='#FF6B6B', alpha=0.6, step='mid')
ax2.set_ylabel('Disorder\n(ground truth)', fontsize=11)
ax2.set_ylim(-0.1, 1.3)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Ordered', 'Disordered'], fontsize=10)
ax2.tick_params(labelsize=10)

# Bottom: predicted disorder
ax3.fill_between(residues_dis, 0, disorder_pred, color='#4ECDC4', alpha=0.6, step='mid')
ax3.set_ylabel('Disorder\n(pLDDT < 50)', fontsize=11)
ax3.set_ylim(-0.1, 1.3)
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['Ordered', 'Disordered'], fontsize=10)
ax3.set_xlabel('Residue index', fontsize=12)
ax3.set_xlim(1, L_dis)
ax3.tick_params(labelsize=10)

# Compute accuracy
accuracy = np.mean(disorder_gt == disorder_pred) * 100
fig.text(0.75, 0.02, f'Prediction accuracy: {accuracy:.1f}%', fontsize=12,
         ha='center', color='#333333',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---

## 8. Domain Boundaries from PAE

### 8.1 Identifying Domains

Sharp transitions in the PAE matrix correspond to domain boundaries. Within a rigid domain, residue pairs have low PAE; across domain boundaries, PAE increases sharply.

A practical approach to detecting domain boundaries is to compute a **boundary score** along the diagonal. For each residue $k$, we compare the average PAE in a local window on one side of the diagonal to the average PAE in a window crossing the diagonal:

$$
B(k) = \frac{1}{|W|} \sum_{(i,j) \in W_{\text{cross}}(k)} \text{PAE}(i,j) - \frac{1}{|W|} \sum_{(i,j) \in W_{\text{within}}(k)} \text{PAE}(i,j)
$$

where $W_{\text{cross}}(k)$ spans the boundary at residue $k$ and $W_{\text{within}}(k)$ stays within one side. Peaks in $B(k)$ indicate domain boundaries.

In [ ]:
# --- Domain boundary detection from PAE ---

np.random.seed(55)
L_dom = 240

# 3-domain protein: domains at 1-70, 71-160, 161-240
boundaries_true = [70, 160]

pae_dom = np.full((L_dom, L_dom), 22.0)

# Domain 1: residues 0-69
pae_dom[:70, :70] = np.random.uniform(1.0, 3.5, (70, 70))
# Domain 2: residues 70-159
pae_dom[70:160, 70:160] = np.random.uniform(1.0, 4.0, (90, 90))
# Domain 3: residues 160-239
pae_dom[160:, 160:] = np.random.uniform(1.0, 3.5, (80, 80))

# Inter-domain: high PAE
pae_dom[:70, 70:160] = np.random.uniform(14, 25, (70, 90))
pae_dom[70:160, :70] = np.random.uniform(14, 25, (90, 70))
pae_dom[:70, 160:] = np.random.uniform(18, 28, (70, 80))
pae_dom[160:, :70] = np.random.uniform(18, 28, (80, 70))
pae_dom[70:160, 160:] = np.random.uniform(14, 24, (90, 80))
pae_dom[160:, 70:160] = np.random.uniform(14, 24, (80, 90))

np.fill_diagonal(pae_dom, 0)
pae_dom = np.clip(pae_dom, 0, 31)

# Compute domain boundary score
window = 15
boundary_score = np.zeros(L_dom)

for k in range(window, L_dom - window):
    # Within-domain average (local block on one side)
    within_block = pae_dom[k-window:k, k-window:k]
    within_avg = np.mean(within_block)
    
    # Cross-domain average (block spanning the boundary)
    cross_block = pae_dom[k-window:k, k:k+window]
    cross_avg = np.mean(cross_block)
    
    boundary_score[k] = cross_avg - within_avg

# Detect peaks in boundary score
# Simple peak detection: local maximum above threshold
threshold_bs = np.mean(boundary_score) + 1.5 * np.std(boundary_score[boundary_score > 0])
detected_boundaries = []
min_distance = 20

for k in range(window, L_dom - window):
    if boundary_score[k] > threshold_bs:
        if (boundary_score[k] >= boundary_score[max(0, k-3):k+4]).all():
            if not detected_boundaries or (k - detected_boundaries[-1]) > min_distance:
                detected_boundaries.append(k)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                                gridspec_kw={'height_ratios': [3, 1.5], 'hspace': 0.15})

# Top: PAE heatmap
im = ax1.imshow(pae_dom, cmap='Greens', vmin=0, vmax=31, origin='lower', aspect='equal')
ax1.set_title('PAE Matrix: 3-Domain Protein (Residues 1--70, 71--160, 161--240)', fontsize=13)
ax1.set_xlabel('Scored residue $j$', fontsize=12)
ax1.set_ylabel('Aligned residue $i$', fontsize=12)

for b in boundaries_true:
    ax1.axvline(x=b, color='white', linestyle='--', linewidth=1.3, alpha=0.7)
    ax1.axhline(y=b, color='white', linestyle='--', linewidth=1.3, alpha=0.7)

ax1.annotate('Domain 1', xy=(30, 30), fontsize=11, color='white', ha='center')
ax1.annotate('Domain 2', xy=(115, 115), fontsize=11, color='white', ha='center')
ax1.annotate('Domain 3', xy=(200, 200), fontsize=11, color='white', ha='center')
plt.colorbar(im, ax=ax1, label='PAE (\u00c5)', shrink=0.7)
ax1.tick_params(labelsize=10)

# Bottom: boundary score
ax2.plot(np.arange(L_dom), boundary_score, color='#333333', linewidth=1.5)
ax2.fill_between(np.arange(L_dom), 0, boundary_score, alpha=0.3, color='#4ECDC4')
ax2.axhline(y=threshold_bs, color='red', linestyle=':', linewidth=1.2, alpha=0.6,
            label=f'Detection threshold ({threshold_bs:.1f})')

# Mark detected boundaries
for b_det in detected_boundaries:
    ax2.axvline(x=b_det, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax2.annotate(f'Boundary\nat {b_det}', xy=(b_det, boundary_score[b_det]),
                 xytext=(b_det + 10, boundary_score[b_det] + 2),
                 fontsize=10, color='red',
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

# Mark true boundaries
for b_true in boundaries_true:
    ax2.axvline(x=b_true, color='blue', linestyle=':', linewidth=1.2, alpha=0.5)

ax2.set_xlabel('Residue index', fontsize=12)
ax2.set_ylabel('Boundary score\n(cross $-$ within PAE)', fontsize=11)
ax2.set_title('Domain Boundary Detection from PAE Gradient', fontsize=13)
ax2.set_xlim(0, L_dom)
ax2.legend(fontsize=10, loc='upper right')
ax2.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

print(f"True domain boundaries: {boundaries_true}")
print(f"Detected boundaries: {detected_boundaries}")

---

## 9. Common Pitfalls and Misinterpretations

Understanding AF2 confidence metrics requires knowing what they do *not* tell you:

### Pitfall 1: High pLDDT does not guarantee correctness

AF2 can produce high-confidence predictions that are structurally incorrect, particularly for:
- Novel folds with no homologues in the training set
- Proteins where the MSA is shallow or misleading
- Conformational states not represented in the PDB

### Pitfall 2: Low pLDDT does not always mean disorder

Low pLDDT can also arise for:
- Flexible but structured regions (e.g., hinge domains)
- Regions with multiple conformations in the crystal
- Membrane-spanning regions in soluble context

### Pitfall 3: PAE directionality

PAE$(i, j) \neq$ PAE$(j, i)$ in general. Aligning at $j$'s frame and measuring error at $i$ is a different question from aligning at $i$'s frame and measuring error at $j$.

### Pitfall 4: pTM for multi-domain proteins

pTM can be misleadingly high for multi-domain proteins where each domain is well-predicted but their relative orientation is wrong. Always check PAE for inter-domain relationships.

In [ ]:
# --- Illustrative examples of pitfalls ---

np.random.seed(33)

fig = plt.figure(figsize=(16, 14))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# --- Pitfall 1: High pLDDT but wrong structure vs Moderate pLDDT but correct ---
L_pit = 80
t_pit = np.linspace(0, 6 * np.pi, L_pit)

# "True" structure: a helix-turn-helix
x_true = np.cos(t_pit) * (1 + 0.02 * np.arange(L_pit))
y_true = np.sin(t_pit) * (1 + 0.02 * np.arange(L_pit))
z_true = np.linspace(0, 8, L_pit)

# "Predicted, high pLDDT but wrong" - different topology
x_wrong = np.cos(t_pit * 1.3) * 2
y_wrong = np.sin(t_pit * 0.7) * 1.5
z_wrong = np.linspace(0, 8, L_pit)
plddt_wrong = np.random.uniform(88, 96, L_pit)  # Confidently wrong

# "Predicted, moderate pLDDT but correct" - close to true
x_correct = x_true + np.random.normal(0, 0.3, L_pit)
y_correct = y_true + np.random.normal(0, 0.3, L_pit)
z_correct = z_true + np.random.normal(0, 0.15, L_pit)
plddt_correct = np.random.uniform(65, 78, L_pit)  # Moderate confidence

# Panel 1: high pLDDT, wrong structure
ax1 = fig.add_subplot(gs[0, 0], projection='3d')
norm_pit = Normalize(vmin=0, vmax=100)
for i in range(L_pit - 1):
    ax1.plot(x_wrong[i:i+2], y_wrong[i:i+2], z_wrong[i:i+2],
             color=cm.RdYlBu(norm_pit(plddt_wrong[i])), linewidth=2.5)
ax1.plot(x_true, y_true, z_true, 'k--', linewidth=1, alpha=0.4, label='True structure')
ax1.set_title('Pitfall 1a: High pLDDT (90+), WRONG fold\n"Confidently incorrect"', fontsize=12)
ax1.legend(fontsize=9)
ax1.tick_params(labelsize=8)
ax1.view_init(elev=20, azim=45)

# Panel 2: moderate pLDDT, correct structure
ax2 = fig.add_subplot(gs[0, 1], projection='3d')
for i in range(L_pit - 1):
    ax2.plot(x_correct[i:i+2], y_correct[i:i+2], z_correct[i:i+2],
             color=cm.RdYlBu(norm_pit(plddt_correct[i])), linewidth=2.5)
ax2.plot(x_true, y_true, z_true, 'k--', linewidth=1, alpha=0.4, label='True structure')
ax2.set_title('Pitfall 1b: Moderate pLDDT (70), CORRECT fold\n"Underconfident but right"', fontsize=12)
ax2.legend(fontsize=9)
ax2.tick_params(labelsize=8)
ax2.view_init(elev=20, azim=45)

# Panel 3: PAE asymmetry
ax3 = fig.add_subplot(gs[1, 0])
L_asym = 60
pae_asym = np.random.uniform(2, 6, (L_asym, L_asym))
# Make it deliberately asymmetric in a region
pae_asym[10:30, 35:55] = np.random.uniform(8, 18, (20, 20))
pae_asym[35:55, 10:30] = np.random.uniform(3, 7, (20, 20))  # Different!
np.fill_diagonal(pae_asym, 0)

im3 = ax3.imshow(pae_asym, cmap='Greens', vmin=0, vmax=20, origin='lower', aspect='equal')
ax3.set_title('Pitfall 3: PAE Asymmetry\nPAE$(i,j) \\neq$ PAE$(j,i)$', fontsize=12)
ax3.set_xlabel('Scored residue $j$', fontsize=11)
ax3.set_ylabel('Aligned residue $i$', fontsize=11)

# Annotate the asymmetric regions
rect1 = plt.Rectangle((35, 10), 20, 20, linewidth=2, edgecolor='red', facecolor='none')
rect2 = plt.Rectangle((10, 35), 20, 20, linewidth=2, edgecolor='blue', facecolor='none')
ax3.add_patch(rect1)
ax3.add_patch(rect2)
ax3.annotate('Higher PAE', xy=(45, 20), fontsize=10, color='red', ha='center')
ax3.annotate('Lower PAE', xy=(20, 45), fontsize=10, color='blue', ha='center')
plt.colorbar(im3, ax=ax3, label='PAE (\u00c5)', shrink=0.8)
ax3.tick_params(labelsize=9)

# Panel 4: pTM misleading for multi-domain
ax4 = fig.add_subplot(gs[1, 1])
scenarios = ['Single domain\n(correct)', 'Multi-domain\n(domains correct,\norientation wrong)',
             'Multi-domain\n(all correct)']
ptm_values = [0.92, 0.78, 0.91]
actual_quality = ['Excellent', 'Poor (misleading)', 'Excellent']
bar_colors_pit = ['#4CAF50', '#FF6B6B', '#4CAF50']

bars = ax4.bar(range(3), ptm_values, color=bar_colors_pit, edgecolor='gray',
               linewidth=0.8, width=0.6)
ax4.set_xticks(range(3))
ax4.set_xticklabels(scenarios, fontsize=10)
ax4.set_ylabel('pTM score', fontsize=12)
ax4.set_title('Pitfall 4: pTM Can Be Misleading\nfor Multi-Domain Proteins', fontsize=12)
ax4.set_ylim(0, 1.1)
ax4.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

for i, (ptm, quality) in enumerate(zip(ptm_values, actual_quality)):
    ax4.text(i, ptm + 0.03, f'pTM={ptm}\n({quality})', ha='center', fontsize=10,
             color='#333333')

ax4.tick_params(labelsize=10)

fig.suptitle('Common Pitfalls in Interpreting AlphaFold2 Confidence Metrics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

## 10. The Complete AlphaFold2 Pipeline: End to End

Having studied each component across 8 notebooks, we can now see the full picture. The diagram below shows the complete AF2 pipeline, from input sequence to confidence-annotated 3D structure, with references to which notebook covers each component.

In [ ]:
# --- Complete AlphaFold2 Pipeline Diagram ---

fig, ax = plt.subplots(figsize=(18, 14))
ax.set_xlim(0, 18)
ax.set_ylim(0, 14)
ax.set_aspect('equal')
ax.axis('off')

# Color scheme
c_input = '#E3F2FD'
c_search = '#FFF3E0'
c_embed = '#E8F5E9'
c_evoformer = '#F3E5F5'
c_structure = '#FFEBEE'
c_output = '#E0F7FA'
c_confidence = '#FFF9C4'

def draw_box(ax, x, y, w, h, text, color, fontsize=11, text_color='#222222'):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='#555555', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, color=text_color, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2, color='#555555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

# --- Row 1: Input ---
draw_box(ax, 7, 12.5, 4, 1, 'Input Sequence\n(amino acid string)', c_input, fontsize=12)
ax.text(11.3, 13, 'Notebook 1', fontsize=9, color='#888888', style='italic')

# Arrows down to searches
draw_arrow(ax, 8, 12.5, 4.5, 11.5)
draw_arrow(ax, 10, 12.5, 13.5, 11.5)

# --- Row 2: Searches ---
draw_box(ax, 2, 10.5, 5, 1, 'MSA Search\n(JackHMMER, HHblits)', c_search, fontsize=11)
ax.text(7.3, 11, 'Notebook 2', fontsize=9, color='#888888', style='italic')

draw_box(ax, 11, 10.5, 5, 1, 'Template Search\n(structural DB)', c_search, fontsize=11)
ax.text(16.3, 11, 'Notebook 2', fontsize=9, color='#888888', style='italic')

# Arrows down to embedding
draw_arrow(ax, 4.5, 10.5, 7.5, 9.5)
draw_arrow(ax, 13.5, 10.5, 10.5, 9.5)

# --- Row 3: Input Embedding ---
draw_box(ax, 5.5, 8.5, 7, 1, 'Input Embedding\nMSA repr. $(N_{\\mathrm{seq}} \\times L \\times c_m)$ + '
         'Pair repr. $(L \\times L \\times c_z)$', c_embed, fontsize=11)
ax.text(12.8, 9, 'Notebook 3', fontsize=9, color='#888888', style='italic')

# Arrow down to Evoformer
draw_arrow(ax, 9, 8.5, 9, 7.8)

# --- Row 4: Evoformer ---
draw_box(ax, 4, 6.3, 10, 1.5, 'Evoformer Stack (48 blocks)\n'
         'MSA row/col attention + Pair triangular attention/updates\n'
         'Outer product mean: MSA -> Pair', c_evoformer, fontsize=11)
ax.text(14.3, 7.3, 'Notebooks 3--4', fontsize=9, color='#888888', style='italic')

# Recycling arrow
ax.annotate('', xy=(3.5, 7.05), xytext=(3.5, 8.8),
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=1.8,
                           connectionstyle='arc3,rad=0.5'))
ax.text(1.5, 7.9, 'Recycling\n(3 iterations)', fontsize=9, color='#9C27B0',
        ha='center', style='italic')

# Arrow down to Structure Module
draw_arrow(ax, 9, 6.3, 9, 5.6)

# --- Row 5: Structure Module ---
draw_box(ax, 4, 4.1, 10, 1.5, 'Structure Module (8 layers)\n'
         'IPA (Invariant Point Attention) + Backbone Update\n'
         'Torsion angle prediction + Side-chain packing', c_structure, fontsize=11)
ax.text(14.3, 5.1, 'Notebooks 5--6', fontsize=9, color='#888888', style='italic')

# Arrows down to outputs
draw_arrow(ax, 7, 4.1, 4.5, 3.3)
draw_arrow(ax, 11, 4.1, 13.5, 3.3)

# --- Row 6: Outputs ---
draw_box(ax, 2, 2.2, 5, 1.1, '3D Structure\nAll-atom coordinates\n$(L \\times 37 \\times 3)$',
         c_output, fontsize=11)
ax.text(7.3, 2.8, 'Notebook 7', fontsize=9, color='#888888', style='italic')

draw_box(ax, 11, 2.2, 5, 1.1, 'Confidence Metrics\npLDDT, PAE, pTM\nModel ranking',
         c_confidence, fontsize=11)
ax.text(16.3, 2.8, 'Notebook 8', fontsize=9, color='#888888', style='italic')

# Loss function annotation
draw_box(ax, 5.5, 0.5, 7, 1, 'Training Losses\nFAPE + distogram + auxiliary losses\n'
         'Confidence heads trained with ground truth', '#F5F5F5', fontsize=10)
ax.text(12.8, 1, 'Notebook 7', fontsize=9, color='#888888', style='italic')

draw_arrow(ax, 4.5, 2.2, 7, 1.5)
draw_arrow(ax, 13.5, 2.2, 11, 1.5)

ax.set_title('The Complete AlphaFold2 Pipeline: End to End', fontsize=16, pad=20)

plt.tight_layout()
plt.show()

---

## 11. Summary: The Full Picture

### 11.1 Recap of All 8 Notebooks

| Notebook | Topic | Key Concepts |
|----------|-------|-------------|
| **1** | Introduction and Protein Fundamentals | Amino acids, backbone geometry, Ramachandran, torsion angles |
| **2** | MSA and Evolutionary Information | Multiple sequence alignment, co-evolution, sequence search |
| **3** | Input Embeddings and the Evoformer | MSA/pair representations, row/column attention, triangular updates |
| **4** | Triangular Attention and Multiplicative Updates | Triangle inequality on distances, axial attention, outer product mean |
| **5** | Invariant Point Attention (IPA) | SE(3)-equivariance, local frames, geometric attention |
| **6** | Structure Module and 3D Coordinate Generation | Iterative backbone update, torsion angles, side-chain placement |
| **7** | Loss Functions and Training | FAPE loss, distogram loss, auxiliary losses, recycling |
| **8** | Confidence Metrics and Interpretation | pLDDT, PAE, pTM, model ranking, disorder, domain detection |

### 11.2 Key Equations Reference

| Metric | Equation | Notebook |
|--------|----------|----------|
| pLDDT | $\text{pLDDT}_i = 100 \times \sum_{b=1}^{50} \frac{b}{50} p_b^{(i)}$ | 8 |
| PAE | $\text{PAE}(i,j) = \mathbb{E}\left[\left\|T_j^{-1} \circ \hat{\mathbf{x}}_i - T_j^{-1} \circ \mathbf{x}_i^{\text{true}}\right\|\right]$ | 8 |
| TM-score | $\text{TM} = \frac{1}{L}\sum_{i=1}^{L} \frac{1}{1+(d_i/d_0)^2}$, $d_0 = 1.24\sqrt[3]{L-15}-1.8$ | 8 |
| FAPE | $L_{\text{FAPE}} = \frac{1}{L^2}\sum_{i,j} \left\| T_i^{-1} \circ \hat{\mathbf{x}}_j - T_i^{\text{true}\,-1} \circ \mathbf{x}_j^{\text{true}} \right\|_{\text{clamp}}$ | 7 |
| IPA | $\text{IPA}(\mathbf{s}, \mathbf{z}, T) = \text{softmax}\left(\frac{1}{\sqrt{c}}\mathbf{q}^T\mathbf{k} + \mathbf{b}_{\text{pair}} + w_L \sum_p \|T_i \circ \hat{\mathbf{q}}_p - T_j \circ \hat{\mathbf{k}}_p\|^2\right)\mathbf{v}$ | 5 |
| Multimer ranking | $\text{score} = 0.8 \times \text{ipTM} + 0.2 \times \text{pTM}$ | 8 |

### 11.3 Final Thoughts

AlphaFold2 represents a paradigm shift in structural biology. Its key innovations include:

1. **End-to-end learning**: From sequence to structure in a single differentiable pipeline.
2. **Evolutionary information**: Deep extraction of co-evolutionary signals via the Evoformer.
3. **Geometric reasoning**: SE(3)-equivariant processing through IPA and frame-based representations.
4. **Calibrated confidence**: pLDDT, PAE, and pTM provide nuanced, interpretable quality estimates.

**Limitations to keep in mind:**

- AF2 predicts a *single static structure*; it does not capture conformational ensembles or dynamics.
- Predictions are only as good as the evolutionary record: orphan proteins with no detectable homologues remain challenging.
- Ligand binding, post-translational modifications, and environmental effects are not modeled.
- The confidence metrics, while generally well-calibrated, can be misleading in edge cases (as discussed in Section 9).

Despite these limitations, AF2 has provided structures for over 200 million proteins and fundamentally changed how we approach problems in biology, drug design, and protein engineering.

---

*This concludes the 8-notebook series on the mathematics and architecture of AlphaFold2.*

In [ ]:
# --- Final summary visualization: confidence metric overview ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: pLDDT summary ---
ax1 = axes[0]
categories = ['Very high\n(> 90)', 'Confident\n(70--90)', 'Low\n(50--70)', 'Very low\n(< 50)']
cat_colors = ['#0066CC', '#87CEEB', '#FFA500', '#FF6B6B']
cat_values = [95, 80, 60, 35]  # representative values
bars1 = ax1.barh(categories, cat_values, color=cat_colors, edgecolor='gray', linewidth=0.8)
ax1.set_xlabel('Representative pLDDT', fontsize=11)
ax1.set_title('pLDDT: Per-Residue Confidence', fontsize=13)
ax1.set_xlim(0, 100)
# Add value labels
for bar, val in zip(bars1, cat_values):
    ax1.text(val + 2, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', fontsize=11)
ax1.tick_params(labelsize=11)

# --- Panel 2: PAE interpretation ---
ax2 = axes[1]
# Small example PAE
L_small = 50
pae_small = np.full((L_small, L_small), 15.0)
pae_small[:25, :25] = np.random.uniform(1, 4, (25, 25))
pae_small[25:, 25:] = np.random.uniform(1, 4, (25, 25))
pae_small[:25, 25:] = np.random.uniform(4, 8, (25, 25))
pae_small[25:, :25] = np.random.uniform(4, 8, (25, 25))
np.fill_diagonal(pae_small, 0)

im2 = ax2.imshow(pae_small, cmap='Greens', vmin=0, vmax=20, origin='lower')
ax2.set_title('PAE: Pairwise Confidence', fontsize=13)
ax2.set_xlabel('Scored residue', fontsize=11)
ax2.set_ylabel('Aligned residue', fontsize=11)
plt.colorbar(im2, ax=ax2, label='Error (\u00c5)', shrink=0.8)
ax2.axvline(x=25, color='white', linestyle='--', linewidth=1.2)
ax2.axhline(y=25, color='white', linestyle='--', linewidth=1.2)
ax2.tick_params(labelsize=10)

# --- Panel 3: pTM / ipTM summary ---
ax3 = axes[2]
metric_names = ['pTM\n(monomer)', 'pTM\n(multimer)', 'ipTM\n(interface)', 'Combined\nscore']
metric_values = [0.91, 0.84, 0.78, 0.8*0.78 + 0.2*0.84]
metric_colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']

bars3 = ax3.bar(metric_names, metric_values, color=metric_colors, edgecolor='gray', linewidth=0.8)
ax3.set_ylabel('Score', fontsize=11)
ax3.set_title('pTM / ipTM: Global Quality', fontsize=13)
ax3.set_ylim(0, 1.1)
ax3.axhline(y=0.5, color='red', linestyle='--', linewidth=1.2, alpha=0.5, label='Fold threshold')

for bar, val in zip(bars3, metric_values):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.2f}', ha='center', fontsize=11)

ax3.legend(fontsize=10)
ax3.tick_params(labelsize=10)

fig.suptitle('AlphaFold2 Confidence Metrics at a Glance', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

print("="*70)
print("  End of Notebook 8: Confidence Metrics and Interpretation")
print("  End of the AlphaFold2 series (8/8)")
print("="*70)